In [0]:
# Load cleaned and enriched data
df = spark.read.table("silver_rides")

df.show(5)

+----------+------------+-------------------+------------+------------+--------------------+----------------+--------+--------+---------------------+----------------------+-------------------+--------------------+----------------+-----------------+-------------+-------------+-------------+---------------+--------------+--------------------+-------------------+----+-----+-----------+----+------------+------------+
|      date|  booking_id|     booking_status| customer_id|vehicle_type|     pickup_location|   drop_location|avg_vtat|avg_ctat|cancelled_by_customer|customer_cancel_reason|cancelled_by_driver|driver_cancel_reason|incomplete_rides|incomplete_reason|booking_value|ride_distance|driver_rating|customer_rating|payment_method|      ingestion_time|   booking_datetime|year|month|day_of_week|hour|is_completed|is_cancelled|
+----------+------------+-------------------+------------+------------+--------------------+----------------+--------+--------+---------------------+-----------------

In [0]:
from pyspark.sql.functions import sum, avg, count

# Overall platform performance
kpi_df = df.select(
    count("*").alias("total_bookings"),
    sum("booking_value").alias("total_revenue"),
    avg("booking_value").alias("avg_booking_value"),
    avg("ride_distance").alias("avg_distance"),
    avg("driver_rating").alias("avg_driver_rating")
)

kpi_df.show()

+--------------+-------------+------------------+------------------+-----------------+
|total_bookings|total_revenue| avg_booking_value|      avg_distance|avg_driver_rating|
+--------------+-------------+------------------+------------------+-----------------+
|        150000|  5.1846183E7|508.29591176470586|24.637011666666663|4.230992473118287|
+--------------+-------------+------------------+------------------+-----------------+



In [0]:
# Understand success vs failure rates
status_df = df.groupBy("booking_status").count()

status_df.show()

+--------------------+-----+
|      booking_status|count|
+--------------------+-----+
|           Completed|93000|
| Cancelled by Driver|27000|
|          Incomplete| 9000|
|     No Driver Found|10500|
|Cancelled by Cust...|10500|
+--------------------+-----+



In [0]:
# Which vehicle generates most revenue?
vehicle_revenue_df = df.groupBy("vehicle_type") \
    .agg(
        sum("booking_value").alias("total_revenue"),
        count("*").alias("total_trips")
    ) \
    .orderBy("total_revenue", ascending=False)

vehicle_revenue_df.show()

+-------------+-------------+-----------+
| vehicle_type|total_revenue|total_trips|
+-------------+-------------+-----------+
|         Auto|  1.2878422E7|      37419|
|      Go Mini|  1.0338496E7|      29806|
|     Go Sedan|    9369719.0|      27141|
|         Bike|    7837697.0|      22517|
|Premier Sedan|    6275332.0|      18111|
|        eBike|    3618485.0|      10557|
|      Uber XL|    1528032.0|       4449|
+-------------+-------------+-----------+



In [0]:
# Peak hours analysis
hourly_df = df.groupBy("hour") \
    .count() \
    .orderBy("hour")

hourly_df.show()

+----+-----+
|hour|count|
+----+-----+
|   0| 1373|
|   1| 1360|
|   2| 1339|
|   3| 1383|
|   4| 1321|
|   5| 2786|
|   6| 4160|
|   7| 5450|
|   8| 6861|
|   9| 8234|
|  10| 9577|
|  11| 8390|
|  12| 7006|
|  13| 5470|
|  14| 7031|
|  15| 8202|
|  16| 9633|
|  17|11044|
|  18|12397|
|  19|11047|
+----+-----+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col

# Customer cancellations
customer_cancel_df = df.filter(col("is_cancelled") == 1) \
    .groupBy("customer_cancel_reason") \
    .count()

customer_cancel_df.show()

# Driver cancellations
driver_cancel_df = df.filter(col("is_cancelled") == 1) \
    .groupBy("driver_cancel_reason") \
    .count()

driver_cancel_df.show()

+----------------------+-----+
|customer_cancel_reason|count|
+----------------------+-----+
|                  NULL|10500|
+----------------------+-----+

+--------------------+-----+
|driver_cancel_reason|count|
+--------------------+-----+
|                NULL|10500|
+--------------------+-----+



In [0]:
# Most active pickup areas
pickup_df = df.groupBy("pickup_location") \
    .count() \
    .orderBy("count", ascending=False)

pickup_df.show()

+----------------+-----+
| pickup_location|count|
+----------------+-----+
|         Khandsa|  949|
| Barakhamba Road|  946|
|           Saket|  931|
|        Badarpur|  921|
|  Pragati Maidan|  920|
|         Madipur|  919|
|           AIIMS|  918|
|        Mehrauli|  915|
|Dwarka Sector 21|  914|
|   Pataudi Chowk|  907|
|    Shivaji Park|  900|
|     Tilak Nagar|  900|
|     Udyog Vihar|  897|
| Greater Kailash|  895|
| Vishwavidyalaya|  895|
|  Kanhaiya Nagar|  895|
|   Tagore Garden|  889|
|   Subhash Chowk|  887|
|          Jasola|  887|
|        Inderlok|  887|
+----------------+-----+
only showing top 20 rows


In [0]:
kpi_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_kpis")

In [0]:
vehicle_revenue_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_vehicle_performance")

In [0]:
hourly_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_hourly_demand")

In [0]:
customer_cancel_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_cancellations")

driver_cancel_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_driver_cancellations")